# 02 · Comparación de modelos en 2017 (incluye modelos entrenados por nosotros)

Qué cambia respecto al notebook 01:
- **Partición por años:** entrenar 2014–2015 · validar 2016 · probar 2017. El examen ahora son las **52 semanas de 2017**, no solo diciembre.
- **Dos modelos entrenados por nosotros:** LightGBM y una red neuronal pequeña (la candidata para el ESP32).
- Compiten contra **Naive semanal** (la vara mínima) y **Chronos** (el ganador del notebook 01).

La explicación de cada decisión está en `README.md`, sección "Parte 2".

**Antes de la corrida completa** pon `PRUEBA_RAPIDA = True` y corre todo: usa solo 20 productos y entrenamientos cortos, para comprobar en un par de minutos que nada falla.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # Keras usa PyTorch (ya instalado por Chronos); TensorFlow solo hará falta en el notebook 03

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import keras
import lightgbm as lgb
from chronos import BaseChronosPipeline

PRUEBA_RAPIDA = False  # True = 20 productos y entrenamientos cortos, solo para comprobar que todo corre

H = 7          # días a predecir
VENTANA = 28   # días de historial reciente que ve cada modelo
ANIO = 364     # "hace un año" = 52 semanas exactas, para caer en el mismo día de la semana
MODELOS_DIR = "../models"
os.makedirs(MODELOS_DIR, exist_ok=True)
keras.utils.set_random_seed(42)  # para que la red salga igual cada vez que se entrena

## 1. Los datos como una matriz

`Y[producto, día]`: una fila por producto-tienda y una columna por día. Es más cómodo que la tabla larga del notebook 01 para recortar ventanas de tiempo.

In [ ]:
raw = pd.read_csv("../data/train.csv", parse_dates=["date"])
raw["unique_id"] = "tienda" + raw.store.astype(str) + "_prod" + raw.item.astype(str)
tabla = raw.pivot(index="unique_id", columns="date", values="sales")
if PRUEBA_RAPIDA:
    tabla = tabla.iloc[:20]

fechas, ids = tabla.columns, tabla.index
assert tabla.notna().all().all() and len(fechas) == (fechas[-1] - fechas[0]).days + 1, "faltan días"
Y = tabla.to_numpy(dtype="float32")
print(f"Y: {Y.shape[0]} productos x {Y.shape[1]} días ({fechas[0].date()} a {fechas[-1].date()})")

## 2. Partición por años y "orígenes"

Un **origen** `t` es el último día que conocemos. Con los datos hasta `t` predecimos `t+1 … t+7`.
Es exactamente lo que hará el sistema real: cada día, "hoy" es el origen.

| Parte | Orígenes | Para qué |
|---|---|---|
| Entrenamiento | todos los días de 2014–2015 | los modelos aprenden aquí |
| Validación | todos los días de 2016 | decidir cuándo dejar de entrenar |
| Prueba | un origen por semana de 2017 (52) | el examen final, se toca una sola vez |

El entrenamiento empieza en 2014 porque cada origen necesita 28 días recientes **más un año** de historia (por la pista "hace un año").

In [ ]:
def idx(fecha):
    return fechas.get_loc(pd.Timestamp(fecha))

primer_origen = ANIO + VENTANA - 1
orig_train = np.arange(primer_origen, idx("2015-12-31") - H + 1)
orig_val = np.arange(idx("2015-12-31"), idx("2016-12-31") - H + 1)
orig_test = np.arange(idx("2017-01-01"), idx("2017-12-31") - H + 1, H)  # mismos cortes semanales que el notebook 01

assert orig_train[-1] + H <= idx("2015-12-31") and orig_val[-1] + H <= idx("2016-12-31")  # ningún objetivo se sale de su año
print(f"orígenes -> entrenamiento: {len(orig_train)} | validación: {len(orig_val)} | prueba: {len(orig_test)} semanas")

## 3. Las pistas (features)

Para cada producto en cada origen armamos **42 números**:

| Pistas | Cuántas | Qué son |
|---|---|---|
| Últimos 28 días | 28 | ventas recientes ÷ escala del producto |
| La misma semana, hace un año | 7 | ventas de esa semana el año pasado ÷ promedio de las 4 semanas previas de ese año |
| Día de la semana | 7 | cuál es el primer día a predecir (uno en 1, los demás en 0) |

**Escala** = promedio de los últimos 28 días. Dividir entre ella hace que un producto que vende 5 y uno que vende 500 se vean iguales para el modelo: aprende **formas**, no tamaños.
Eso, y no usar el nombre del producto ni la fecha del año, es lo que permite aplicarlo a **cualquier inventario**.

Lo que el modelo aprende a predecir: las ventas de los próximos 7 días ÷ escala. Al final se multiplica de vuelta.

In [ ]:
def armar_pistas(Y, t):
    """Pistas de todos los productos en el origen t (último día conocido). Devuelve X (productos x 42) y la escala."""
    reciente = Y[:, t - VENTANA + 1 : t + 1]
    # shortcut: un producto que casi no vende (promedio < 1/día) no se normaliza; revisar con datos reales intermitentes
    escala = np.maximum(reciente.mean(axis=1, keepdims=True), 1)
    hace_un_anio = Y[:, t - ANIO + 1 : t - ANIO + 1 + H]
    base_anio = np.maximum(Y[:, t - ANIO - VENTANA + 1 : t - ANIO + 1].mean(axis=1, keepdims=True), 1)
    dia_semana = np.zeros((len(Y), 7), dtype="float32")
    dia_semana[:, (fechas[t].dayofweek + 1) % 7] = 1
    return np.hstack([reciente / escala, hace_un_anio / base_anio, dia_semana]), escala


def armar_tabla(origenes):
    """Apila las pistas de varios orígenes. Filas: origen 1 (todos los productos), origen 2, ..."""
    Xs, ys, escalas = [], [], []
    for t in origenes:
        X, escala = armar_pistas(Y, t)
        Xs.append(X)
        ys.append(Y[:, t + 1 : t + 1 + H] / escala)
        escalas.append(escala)
    return np.vstack(Xs), np.vstack(ys), np.vstack(escalas)


Xtr, ytr, _ = armar_tabla(orig_train)
Xva, yva, _ = armar_tabla(orig_val)
Xte, _, escala_te = armar_tabla(orig_test)
print("entrenamiento:", Xtr.shape, "| validación:", Xva.shape, "| prueba:", Xte.shape)

n_sem, n_prod = len(orig_test), len(Y)
real = np.stack([Y[:, t + 1 : t + 1 + H] for t in orig_test])  # (semanas, productos, 7 días): las respuestas del examen
pred = {}

## 4. Modelos de referencia

- **Naive semanal:** copia la semana anterior. Si un modelo no le gana a esto, no sirve.
- **Chronos-Bolt small:** el ganador del notebook 01. No se entrena: en cada semana de 2017 recibe todo el historial hasta ese día.

Chronos es lo más lento de este notebook: 52 semanas × 500 productos.

In [ ]:
pred["Naive_semanal"] = np.stack([Y[:, t - H + 1 : t + 1] for t in orig_test])

chronos = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small", device_map="cpu", torch_dtype=torch.float32)
semanas = []
for t in orig_test:
    q, _ = chronos.predict_quantiles(torch.tensor(Y[:, : t + 1]), prediction_length=H, quantile_levels=[0.5])
    semanas.append(q[..., 0].numpy())
pred["Chronos"] = np.stack(semanas)

## 5. LightGBM (entrenado por nosotros)

Un modelo por cada día a predecir (día 1, día 2 … día 7): **7 modelos**. Cada uno es una cadena de árboles de decisión donde cada árbol corrige al anterior.

- `objective="l1"`: aprende a minimizar el error absoluto, o sea, a predecir la **mediana** (el P50, igual que Chronos).
- **Early stopping:** agrega árboles mientras mejore en 2016 (validación); cuando deja de mejorar 50 rondas seguidas, para. Así no memoriza.
- Al final se **reentrena con 2014–2016** usando ese número de árboles, para que también aprenda de 2016.

In [ ]:
params = dict(objective="l1", learning_rate=0.05, num_leaves=31, min_data_in_leaf=200,
              feature_fraction=0.8, verbose=-1, seed=42)
rondas = 50 if PRUEBA_RAPIDA else 2000

lgb_modelos = []
for k in range(H):
    m = lgb.train(params, lgb.Dataset(Xtr, ytr[:, k].copy()), rondas,  # .copy(): LightGBM avisa si recibe una columna "rebanada"
                  valid_sets=[lgb.Dataset(Xva, yva[:, k].copy())], callbacks=[lgb.early_stopping(50, verbose=False)])
    final = lgb.train(params, lgb.Dataset(np.vstack([Xtr, Xva]), np.concatenate([ytr[:, k], yva[:, k]])), m.best_iteration)
    final.save_model(f"{MODELOS_DIR}/lightgbm_dia{k + 1}.txt")
    lgb_modelos.append(final)
    print(f"día {k + 1}: {m.best_iteration} árboles")

pred["LightGBM"] = (np.column_stack([m.predict(Xte) for m in lgb_modelos]) * escala_te).reshape(n_sem, n_prod, H)

## 6. Red neuronal pequeña (entrenada por nosotros, candidata para el ESP32)

Un **perceptrón multicapa** (MLP): 42 pistas → 64 neuronas → 32 neuronas → 7 salidas (una por día).
Tiene ~5,000 parámetros: unos 20 KB en float32 y ~5 KB en int8. Cabe de sobra en el ESP32-S3.

- `loss="mae"`: igual que LightGBM, aprende la mediana.
- **Early stopping:** si el error en 2016 no mejora en 5 épocas seguidas, para y se queda con la mejor.
- Luego se **reentrena desde cero con 2014–2016** durante ese mismo número de épocas.

In [ ]:
def crear_red():
    red = keras.Sequential([
        keras.Input(shape=(Xtr.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(H),
    ])
    red.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mae")
    return red

epocas = 2 if PRUEBA_RAPIDA else 100
red = crear_red()
hist = red.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epocas, batch_size=512, verbose=0,
               callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
mejor = int(np.argmin(hist.history["val_loss"])) + 1

red = crear_red()
red.fit(np.vstack([Xtr, Xva]), np.vstack([ytr, yva]), epochs=mejor, batch_size=512, verbose=0)
red.save(f"{MODELOS_DIR}/red.keras")
print(f"{red.count_params():,} parámetros | mejor época: {mejor}")

pred["Red_neuronal"] = (red.predict(Xte, verbose=0) * escala_te).reshape(n_sem, n_prod, H)

plt.figure(figsize=(8, 3))
plt.plot(hist.history["loss"], label="entrenamiento (2014-2015)")
plt.plot(hist.history["val_loss"], label="validación (2016)")
plt.axvline(mejor - 1, color="gray", ls=":", label="mejor época")
plt.xlabel("época"); plt.ylabel("error (MAE normalizado)"); plt.title("Curva de aprendizaje"); plt.legend()

## 7. Comparación final

Mismas métricas que el notebook 01 (ver README, sección 3), ahora sobre las 52 semanas de 2017.
**Tamaño** es lo que ocupa el modelo; es lo que decide si cabe en el ESP32.

In [ ]:
assert all(p.shape == real.shape for p in pred.values())

def metricas(p):
    err = p - real
    return {"MAE (unidades/día)": np.abs(err).mean(),
            "WAPE": np.abs(err).sum() / real.sum(),
            "Sesgo": err.sum() / real.sum(),
            "WAPE semanal": np.abs(err.sum(axis=2)).sum() / real.sum()}

tam_kb = {"Naive_semanal": 0,
          "Chronos": 183_000,  # lo que pesa la descarga del modelo
          "LightGBM": sum(os.path.getsize(f"{MODELOS_DIR}/lightgbm_dia{k + 1}.txt") for k in range(H)) / 1024,  # archivo de texto; en C ocupa menos
          "Red_neuronal": red.count_params() * 4 / 1024}  # float32 = 4 bytes por parámetro; en int8, ~1/4

resultados = pd.DataFrame({m: metricas(p) for m, p in pred.items()}).T
resultados["Tamaño (KB)"] = pd.Series(tam_kb)
print(f"Control: MAE del naive en diciembre = {np.abs(pred['Naive_semanal'][-4:] - real[-4:]).mean():.2f} (en el notebook 01 dio 8.74)")
resultados.sort_values("WAPE semanal").style.format(
    {"MAE (unidades/día)": "{:.2f}", "WAPE": "{:.1%}", "Sesgo": "{:+.1%}", "WAPE semanal": "{:.1%}", "Tamaño (KB)": "{:,.0f}"})

### ¿En qué meses falla cada uno?

Error semanal promedio de cada mes. Sirve para ver si un modelo se cae en los cambios de temporada (como diciembre en el notebook 01).

In [ ]:
def wape_por_semana(p):
    return np.abs(p.sum(axis=2) - real.sum(axis=2)).sum(axis=1) / real.sum(axis=(1, 2))

meses = fechas[orig_test + 1].month
por_mes = pd.DataFrame({m: wape_por_semana(p) for m, p in pred.items()}, index=meses).groupby(level=0).mean()
por_mes.plot(marker="o", figsize=(10, 4), title="WAPE semanal promedio por mes (2017)", xlabel="mes", ylabel="WAPE");

### Un producto, todo 2017

Total de cada semana: lo real contra lo que predijo cada modelo.

In [ ]:
i = 0
plt.figure(figsize=(12, 4))
plt.plot(real[:, i].sum(axis=1), "k-o", ms=3, label="real")
for m, p in pred.items():
    plt.plot(p[:, i].sum(axis=1), "--", label=m)
plt.title(f"{ids[i]}: total de cada semana de 2017"); plt.xlabel("semana"); plt.ylabel("unidades"); plt.legend(ncol=5, fontsize=8);

## 8. Siguiente paso

Si la red neuronal le gana al naive semanal, el **notebook 03** la convierte a int8 (TensorFlow Lite), **vuelve a medirla** con estas mismas 52 semanas y genera el archivo para el ESP32.